### BP batch processing
Runs the segmentation and feature-extraction pipeline on all images in `data_folder`, saves one CSV per image under `results/experiment_id`. No filtering or visualization.

### Config: data folder, markers, output directory

In [1]:
import os
from pathlib import Path

from utils import list_images, read_image, extract_img_metadata

# Path to folder containing images
data_folder = r"X:\Shirin\RCDAnalysis_Alberto\SK0047_Exp01-02"

# Markers for intensity analysis: (channel_name, channel_nr), 0-based
markers = [("SD_DAPI", 1), ("SD_GFP", 2), ("SD_RFP", 3), ("SD_AF647", 4)]

images = list_images(data_folder, format="nd2")
experiment_id = Path(data_folder).name
out_dir = Path("results") / experiment_id
os.makedirs(out_dir, exist_ok=True)
print(f"Found {len(images)} images. Output: {out_dir}")

Found 20 images. Output: results\SK0047_Exp01-02


### Imports and model load (run once)

In [2]:
from cellpose import models, core
from skimage.segmentation import clear_border
from skimage.measure import regionprops_table
import pandas as pd
import numpy as np

import logging
logging.getLogger("tifffile").setLevel(logging.ERROR)

if core.use_gpu() == False:
    raise ImportError("No GPU access, change your runtime")

model = models.CellposeModel(gpu=True)



Welcome to CellposeSAM, cellpose v
cellpose version: 	4.0.6 
platform:       	win32 
python version: 	3.10.18 
torch version:  	2.5.0! The neural network component of
CPSAM is much larger than in previous versions and CPU excution is slow. 
We encourage users to use GPU/MPS if available. 




### Morphology and intensity property lists

In [3]:
morphology_properties = [
    "label",
    "area",
    "area_bbox",
    "area_convex",
    "area_filled",
    "axis_major_length",
    "axis_minor_length",
    "equivalent_diameter_area",
    "euler_number",
    "extent",
    "feret_diameter_max",
    "solidity",
    "inertia_tensor_eigvals",
]

intensity_properties = [
    "label",
    "intensity_mean",
    "intensity_min",
    "intensity_max",
    "intensity_std",
]

### Process all images and save CSVs

In [4]:
for i, img_filepath in enumerate(images):
    print(f"Processing {i + 1}/{len(images)}: {Path(img_filepath).name}")

    img, filename = read_image(img_filepath, log=False)
    descriptor_dict = extract_img_metadata(img_filepath, verbose=False)

    cell_labels, flows, styles = model.eval(img[0], niter=1000)
    cell_labels = clear_border(cell_labels)

    props_morphology = regionprops_table(
        label_image=cell_labels,
        properties=morphology_properties,
    )
    props_df = pd.DataFrame(props_morphology)

    for marker_name, ch_nr in markers:
        props = regionprops_table(
            label_image=cell_labels,
            intensity_image=img[ch_nr],
            properties=intensity_properties,
        )
        intensity_df = pd.DataFrame(props)

        prefix = f"{marker_name}"
        rename_map = {"label": "label"}
        for prop in intensity_properties:
            if prop == "label":
                continue
            if prop.startswith("intensity_"):
                suffix = prop.replace("intensity_", "")
                rename_map[prop] = f"{prefix}_{suffix}_int"
        intensity_df.rename(columns=rename_map, inplace=True)

        mean_col = rename_map["intensity_mean"]
        max_col = rename_map["intensity_max"]
        intensity_df[f"{prefix}_max_mean_ratio"] = (
            intensity_df[max_col] / intensity_df[mean_col].replace(0, np.nan)
        )

        props_df = props_df.merge(intensity_df, on="label")
        props_df[f"{prefix}_sum_int"] = props_df[mean_col] * props_df["area"]

    insertion_position = 0
    for key, value in descriptor_dict.items():
        props_df.insert(insertion_position, key, value)
        insertion_position += 1

    csv_path = out_dir / f"{descriptor_dict['well_id']}.csv"
    props_df.to_csv(csv_path, index=False)
    print(f"  -> {csv_path}")

print("Done.")

Processing 1/20: 251120_SK_SK0047_Exp1_WellB2_Pos01_after4h.nd2
  -> results\SK0047_Exp01-02\B2.csv
Processing 2/20: 251120_SK_SK0047_Exp1_WellB3_Pos06_after4h003.nd2
  -> results\SK0047_Exp01-02\B3.csv
Processing 3/20: 251120_SK_SK0047_Exp1_WellC2_Pos02_after4h.nd2
  -> results\SK0047_Exp01-02\C2.csv
Processing 4/20: 251120_SK_SK0047_Exp1_WellC3_Pos07_after4h004.nd2
  -> results\SK0047_Exp01-02\C3.csv
Processing 5/20: 251120_SK_SK0047_Exp1_WellD2_Pos03_after4h.nd2
  -> results\SK0047_Exp01-02\D2.csv
Processing 6/20: 251120_SK_SK0047_Exp1_WellD3_Pos08_after4h005.nd2
  -> results\SK0047_Exp01-02\D3.csv
Processing 7/20: 251120_SK_SK0047_Exp1_WellE2_Pos04_after4h001.nd2
  -> results\SK0047_Exp01-02\E2.csv
Processing 8/20: 251120_SK_SK0047_Exp1_WellE3_Pos09_after4h006.nd2
  -> results\SK0047_Exp01-02\E3.csv
Processing 9/20: 251120_SK_SK0047_Exp1_WellF2_Pos05_after4h002.nd2
  -> results\SK0047_Exp01-02\F2.csv
Processing 10/20: 251120_SK_SK0047_Exp1_WellF3_Pos10_after4h007.nd2
  -> results\S